In [99]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

df = pd.read_csv("data/ffl_list.txt", sep="\t")
zips = pd.read_csv("data/2013_us_zips.csv")
df = df[df["LIC_TYPE"] == 1 | 2 | 7]
df = df[["MAIL_ZIP_CODE", "LICENSE_NAME"]].merge(right= zips, left_on="MAIL_ZIP_CODE", right_on="ZIP").drop(columns="ZIP")

f = df.groupby("MAIL_ZIP_CODE").agg("first")
df = df.groupby("MAIL_ZIP_CODE").agg("count")[["LAT"]].rename(columns={"LAT":"count"})
df = df.merge(f, right_index=True, left_index=True)
df["count"] = np.log(df["count"])
df.head()

,count,LICENSE_NAME,LAT,LNG
MAIL_ZIP_CODE,,,,
602,0.000000,EMPRESAS SALAZAR LLC,18.361945,-67.175597
623,0.000000,"WEST SHOOTING SUPPLY, LLC",18.083361,-67.153897
662,0.000000,ANGOSTURA INC,18.468320,-67.015781
682,0.693147,"SALAS, CHRISTOPHER SALAZAR",18.221464,-67.156039
685,0.693147,PEPINO GUN GALLERY INC,18.332929,-66.959689


In [100]:
df[df["count"] == df["count"].max()]

,count,LICENSE_NAME,LAT,LNG
MAIL_ZIP_CODE,,,,
85142,3.433987,RIGID DEFENSE LLC,33.19999,-111.636294


In [101]:

fig = px.scatter_geo(df,lat='LAT',lon='LNG', hover_name="LICENSE_NAME", color="count")
fig.update_traces(marker=dict(size=3))
fig.update_layout(title = 'Licensed Gun Realtors', title_x=0.5)
fig.show()

fig = go.Figure(data=go.Scattergeo(
        lon = df['LNG'],
        lat = df['LAT'],
        text = df['LICENSE_NAME'],
        mode = 'markers',
        marker_color = df['count'],
        marker = dict(
            size = 3,
            opacity = 0.8,
        colorbar=dict(
                title=dict(
                    text="Log(Number of Licensed Realtors)")
                    )
            )
        )
)

fig.update_layout(
        title = 'Licensed Firearm Realtors in the US, by Zip Code',
        geo_scope='usa',
    )
fig.show()

In [102]:
df = pd.read_csv("data/ffl_list.txt", sep="\t")
zips = pd.read_csv("data/2013_us_zips.csv")
df = df[df["LIC_TYPE"] == 1 | 2 | 7]
df = df[["MAIL_ZIP_CODE", "LICENSE_NAME"]].merge(right= zips, left_on="MAIL_ZIP_CODE", right_on="ZIP").drop(columns="ZIP")
# add jitter instead of grouping
jitter_lat = np.random.normal(0,1,len(df)) / 10
jitter_lon = np.random.normal(0,1,len(df)) / 10
df["LAT"] = df["LAT"] + jitter_lat
df["LNG"] = df["LNG"] + jitter_lon
df.head()


,MAIL_ZIP_CODE,LICENSE_NAME,LAT,LNG
0,602,EMPRESAS SALAZAR LLC,18.374466,-67.215658
1,623,"WEST SHOOTING SUPPLY, LLC",18.061682,-66.930939
2,662,ANGOSTURA INC,18.570711,-67.075222
3,685,PEPINO GUN GALLERY INC,18.275302,-67.077151
4,685,PEPINO GUN GALLERY INC,18.223564,-67.156917


In [103]:

fig = go.Figure(data=go.Scattergeo(
        lon = df['LNG'],
        lat = df['LAT'],
        text = df['LICENSE_NAME'],
        mode = 'markers',
        marker = dict(
            size = 3,
            opacity = 0.8,
        )
        ))

fig.update_layout(
        title = 'Licensed Firearm Realtors in the US',
        geo_scope='usa',
    )
fig.show()


In [104]:
# plot MJ data
mj = pd.read_csv("data/mother_jones.csv")[["location", "fatalities", "injured", "year", "case"]]
# separate location
pat = ", ([A-Z]+[a-z]+)"
states = mj["location"].str.extract(pat)
mj["state"] = states
pat = "([A-Za-z ]+),"
cities = mj["location"].str.extract(pat)
mj["city"] = cities    
mj.head()


,location,fatalities,injured,year,case,state,city
0,"Winder, Georgia",4,9,2024,Apalachee High School shooting,Georgia,Winder
1,"Fordyce, Arkansas",4,10,2024,Arkansas grocery store shooting,Arkansas,Fordyce
2,"Las Vegas, Nevada",3,1,2023,UNLV shooting,Nevada,Las Vegas
3,"Lewiston, Maine",18,13,2023,Maine bowling alley and bar shootings,Maine,Lewiston
4,"Jacksonville, Florida",3,0,2023,Jacksonville Dollar General store shooting,Florida,Jacksonville


In [105]:
cities = pd.read_csv("data/uscities.csv")[["city", "state_name", "lat", "lng", "population", "density"]]
cities.head()

,city,state_name,lat,lng,population,density
0,New York,New York,40.6943,-73.9249,18832416,10943.7
1,Los Angeles,California,34.1141,-118.4068,11885717,3165.8
2,Chicago,Illinois,41.8375,-87.6866,8489066,4590.3
3,Miami,Florida,25.7840,-80.2101,6113982,4791.1
4,Houston,Texas,29.7860,-95.3885,6046392,1386.5


In [106]:
mj = mj.merge(right=cities, right_on=["city", "state_name"], left_on=["city", "state"])
mj.head()

,location,fatalities,injured,year,case,state,city,state_name,lat,lng,population,density
0,"Winder, Georgia",4,9,2024,Apalachee High School shooting,Georgia,Winder,Georgia,33.9917,-83.7218,18847,505.4
1,"Fordyce, Arkansas",4,10,2024,Arkansas grocery store shooting,Arkansas,Fordyce,Arkansas,33.8182,-92.4174,3320,190.6
2,"Las Vegas, Nevada",3,1,2023,UNLV shooting,Nevada,Las Vegas,Nevada,36.2333,-115.2654,2256509,1771.5
3,"Las Vegas, Nevada",60,546,2017,Las Vegas Strip massacre,Nevada,Las Vegas,Nevada,36.2333,-115.2654,2256509,1771.5
4,"Lewiston, Maine",18,13,2023,Maine bowling alley and bar shootings,Maine,Lewiston,Maine,44.0915,-70.1681,37886,428.4


In [107]:
fig = go.Figure(data=go.Scattergeo(
        lon = mj['lng'],
        lat = mj['lat'],
        text = mj['case'],
        mode = 'markers',
        marker = dict(
            size = np.sqrt(mj["fatalities"] + mj["injured"])*2,
            opacity = 0.8,
            color = mj['year'],
            colorbar=dict(
                title=dict(
                    text="Event Year")
                    )
        )))

fig.update_layout(
        title = 'Mass Shooting Events in the United States, Location vs Sqrt(fatalities + injured)',
        geo_scope='usa',
    )
fig.show()